# Experiment 9
## Autoregressive Model – PixelCNN
**Aim:** Implement a simplified PixelCNN autoregressive model using PyTorch.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

### Step 2 – Load MNIST Dataset

In [ ]:
transform    = transforms.Compose([transforms.ToTensor()])
train_data   = datasets.MNIST('./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
print('MNIST loaded:', len(train_data), 'images of size 28x28')

### Step 3 – Define Masked Convolution (Core of PixelCNN)
The masked convolution ensures each pixel only depends on pixels that came before it (autoregressive property).

In [ ]:
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mask_type = mask_type
        mask = torch.ones_like(self.weight.data)
        _, _, H, W = self.weight.size()
        mask[:, :, H//2, W//2 + (1 if mask_type == 'B' else 0):] = 0  # zero out future pixels
        mask[:, :, H//2 + 1:, :] = 0
        self.register_buffer('mask', mask)

    def forward(self, x):
        self.weight.data *= self.mask
        return super().forward(x)

### Step 4 – Build PixelCNN Model

In [ ]:
class PixelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            MaskedConv2d('A', 1,  64, 7, padding=3), nn.ReLU(),
            MaskedConv2d('B', 64, 64, 7, padding=3), nn.ReLU(),
            MaskedConv2d('B', 64, 64, 7, padding=3), nn.ReLU(),
            nn.Conv2d(64, 256, 1)   # 256 output channels → pixel values 0-255
        )
    def forward(self, x): return self.layers(x)

model = PixelCNN()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

### Step 5 – Train the PixelCNN

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses    = []
for epoch in range(3):
    total_loss = 0
    for imgs, _ in train_loader:
        targets = (imgs * 255).long().squeeze(1)   # pixel values 0-255
        logits  = model(imgs)
        loss    = F.cross_entropy(logits, targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    avg = total_loss / len(train_loader)
    losses.append(avg)
    print(f'Epoch {epoch+1}/3 | Loss: {avg:.4f}')

### Step 6 – Generate Images Autoregressively

In [ ]:
model.eval()
generated = torch.zeros(4, 1, 28, 28)         # start with blank canvas
with torch.no_grad():
    for row in range(28):
        for col in range(28):
            logits = model(generated)
            probs  = torch.softmax(logits[:, :, row, col], dim=1)
            pixel  = torch.multinomial(probs, 1).float() / 255.0
            generated[:, 0, row, col] = pixel.squeeze()

fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].squeeze().numpy(), cmap='gray'); ax.axis('off')
plt.suptitle('PixelCNN Generated Images'); plt.show()

### Result
PixelCNN was implemented using masked convolutions to enforce the autoregressive property. Images were generated pixel-by-pixel from left to right, top to bottom.